In [31]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os

In [32]:
raw_ihs_poverty_2010 = pd.read_csv(r'd:\GG\source\householdpoverty_10.CSV')
raw_ihs_weight_2010 = pd.read_csv(r'd:\GG\source\householdweight_10.CSV')
raw_ihs_weight_2015 = pd.read_csv(r'd:\GG\source\householdweight_15.CSV')
raw_ihs_poverty_2015 = pd.read_csv(r'd:\GG\source\householdpoverty_15.CSV')
raw_dhs_2020 = pd.read_stata(r'd:\GG\source\household_19_20.DTA')
raw_findex_2021 = pd.read_csv(r'd:\GG\source\connectivity_21.csv')
raw_findex_2024 = pd.read_csv(r'd:\GG\source\connectivity_24.csv')

In [33]:
# aggregate economic rank
from numpy import nan as NA

# 2010
df_2010 = pd.DataFrame()
df_2010['hid'] = raw_ihs_poverty_2010['hid']
df_2010['survey_year'] = 2010
df_2010['data_source'] = 'IHS'
df_2010['lga'] = raw_ihs_poverty_2010['lga']

weight_lookup = raw_ihs_weight_2010[['hid', 'weightslga']].drop_duplicates(subset=['hid'])
df_2010 = df_2010.merge(weight_lookup, on='hid', how='left')
df_2010['weightslga'] = df_2010['weightslga'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2010.drop_duplicates('hid').set_index('hid')['s11q2']
df_2010['hh_income'] = df_2010['hid'].astype(int).map(poverty_map)

# 2015
df_2015 = pd.DataFrame()
df_2015['hh_id'] = raw_ihs_poverty_2015['hid']
df_2015['survey_year'] = 2015
df_2015['data_source'] = 'IHS'
df_2015['eanum'] = raw_ihs_poverty_2015['eanum']

weight_lookup = raw_ihs_weight_2015[['eanum', 'hhweight']].drop_duplicates(subset=['eanum'])
df_2015 = df_2015.merge(weight_lookup, on='eanum', how='left')
df_2015['hh_weight'] = df_2015['hhweight'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2015.drop_duplicates('hid').set_index('hid')['s13q3']
df_2015['hh_income'] = df_2015['hh_id'].astype(int).map(poverty_map)

# 2020
df_2020 = pd.DataFrame()
df_2020['hh_id'] = raw_dhs_2020['hhid']
df_2020['survey_year'] = raw_dhs_2020['hv007']
df_2020['hh_income'] = raw_dhs_2020['hv270']

df_2020['data_source'] = 'DHS'
df_2020['weight'] = raw_dhs_2020['hv005']

quintile_labels = {'poorest': 1, 'poorer': 2, 'middle': 3, 'richer': 4, 'richest': 5}
df_2020['hh_income'] = df_2020['hh_income'].map(quintile_labels)

# 2021
df_2021 = pd.DataFrame()
df_2021['hh_id'] = raw_findex_2021.index.map(lambda x: f"findex_21_{x}")
df_2021['survey_year'] = 2021
df_2021['data_source'] = 'Findex'
df_2021['hh_income'] = raw_findex_2021['inc_q']
df_2021['weight'] = (raw_findex_2021['wgt']).fillna(NA)

# 2024
df_2024 = pd.DataFrame()
df_2024['hh_id'] = raw_findex_2024.index.map(lambda x: f"findex_24_{x}")
df_2024['survey_year'] = 2024
df_2024['data_source'] = 'Findex'
df_2024['hh_income'] = raw_findex_2024['inc_q']
df_2024['weight'] = (raw_findex_2024['wgt']).fillna(NA)

In [34]:
# econ rank construct
df_2010['hh_econ_rank'] = (df_2010.sort_values('hh_income')['weightslga'].cumsum() - 0.5 * df_2010['weightslga']) / df_2010['weightslga'].sum() * 100
df_2015['hh_econ_rank'] = (df_2015.sort_values('hh_income')['hh_weight'].cumsum() - 0.5 * df_2015['hh_weight']) / df_2015['hh_weight'].sum() * 100
df_2020['hh_econ_rank'] = (df_2020.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2020['weight']) / df_2020['weight'].sum() * 100
df_2021['hh_econ_rank'] = (df_2021.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2021['weight']) / df_2021['weight'].sum() * 100
df_2024['hh_econ_rank'] = (df_2024.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2024['weight']) / df_2024['weight'].sum() * 100

In [35]:
# concat all
df_2010.rename(columns={'hid': 'hh_id', 'weightslga': 'weight'}, inplace=True)
df_2015.rename(columns={'hh_weight': 'weight'}, inplace=True)

target_cols = ['hh_id', 'survey_year', 'data_source', 'hh_income', 'weight', 'hh_econ_rank']

df_2010 = df_2010[target_cols]
df_2015 = df_2015[target_cols]
df_2020 = df_2020[target_cols]
df_2021 = df_2021[target_cols]
df_2024 = df_2024[target_cols]

dfs = [df_2010, df_2015, df_2020, df_2021, df_2024]
for df in dfs:
    df['hh_id'] = df['hh_id'].astype(str)

df_panel = pd.concat(dfs, ignore_index=True)
df_panel = df_panel.dropna(subset=['hh_id', 'hh_income', 'weight'])

df_panel['unique_id'] = (
    df_panel['data_source'] + '_' + 
    df_panel['survey_year'].astype(str) + '_' + 
    df_panel['hh_id']
)

df_panel.set_index('unique_id', inplace=True)

display(df_panel.tail())

,hh_id,survey_year,data_source,hh_income,weight,hh_econ_rank
unique_id,,,,,,
Findex_2024_findex_24_1003,findex_24_1003,2024,Findex,5.0,0.328288,81.159517
Findex_2024_findex_24_1004,findex_24_1004,2024,Findex,2.0,0.777492,20.392583
Findex_2024_findex_24_1005,findex_24_1005,2024,Findex,1.0,0.671328,19.898279
Findex_2024_findex_24_1006,findex_24_1006,2024,Findex,4.0,0.483620,79.927619
Findex_2024_findex_24_1007,findex_24_1007,2024,Findex,5.0,0.798636,99.960385


In [36]:
# macro data
df = pd.read_csv(r'd:\GG\source\monthly_foodprices.CSV')
# Create monthly period from year and month columns
df['month_period'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str)).dt.to_period('M')

# Aggregate the composite national food price index and calculate MoM growth
df_macro = df.groupby('month_period')['o_food_price_index'].mean().to_frame()
df_macro['index_MoM'] = df_macro['o_food_price_index'].pct_change()

display(df_macro.head())

C:\Users\Dell\AppData\Local\Temp\ipykernel_24064\788947014.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['month_period'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str)).dt.to_period('M')


,o_food_price_index,index_MoM
month_period,,
2007-01,0.476897,NaN
2007-02,0.484483,0.015907
2007-03,0.494828,0.021352
2007-04,0.508621,0.027875
2007-05,0.530690,0.043390


In [37]:
# Economic Rank: Normalize and Extract Average 
from scipy.stats import norm

# 1. Transform Rank to Continuous Space
df_panel['rank_pct'] = df_panel['hh_econ_rank'].clip(0.1, 99.9) / 100
df_panel['rank_continuous'] = norm.ppf(df_panel['rank_pct'])

# 2. Compute Weighted Average per Year
def weighted_mean(group):
    return np.average(group['rank_continuous'], weights=group['weight'])

df_cohort = df_panel.groupby('survey_year').apply(weighted_mean).reset_index(name='avg_rank_continuous')

# 3. Anchor Annual Survey to a Single Month (e.g., June)
df_cohort['anchor_month'] = 6

# 4. Prepare the Monthly Timeline (using your existing df_macro)
df_timeline = df_macro.reset_index().copy()
df_timeline['merge_year'] = df_timeline['month_period'].dt.year
df_timeline['merge_month'] = df_timeline['month_period'].dt.month

# 5. Merge the Survey Data onto the Monthly Timeline
df_timeline = df_timeline.merge(
    df_cohort[['survey_year', 'anchor_month', 'avg_rank_continuous']], 
    left_on=['merge_year', 'merge_month'], 
    right_on=['survey_year', 'anchor_month'], 
    how='left'
)

# 6. Clean up temporary merge columns
df_timeline.set_index('month_period', inplace=True)
df_timeline.drop(columns=['merge_year', 'merge_month', 'survey_year', 'anchor_month'], inplace=True)

# Display only the months where the survey data successfully attached (June of survey years)
display(df_timeline[df_timeline['avg_rank_continuous'].notna()])

,o_food_price_index,index_MoM,avg_rank_continuous
month_period,,,
2010-06,0.547586,0.003793,-0.146866
2015-06,0.925517,-0.007763,-0.000002
2019-06,1.040690,0.006335,0.576763
2020-06,1.139310,0.055928,-0.243251
2021-06,1.238966,0.015546,0.000018
2024-06,2.026207,0.014153,-0.000076


In [40]:
# Fit State-Space Model
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1. Clean Exogenous Driver
# Statsmodels skips NaNs in Endog natively, but crashes if Exog has NaNs. 
df_timeline['index_MoM'] = df_timeline['index_MoM'].fillna(0)

# 2. Define Variables
endog = df_timeline['avg_rank_continuous']
exog = df_timeline['index_MoM']

# 3. Fit State-Space Model (AR(1) with Exogenous Food Price Driver)
model = SARIMAX(endog, exog=exog, order=(1, 0, 0), trend='c')
model_fit = model.fit(disp=False)

# 4. Extract Predictions and Impute Pseudo Data
df_timeline['imputed_continuous'] = model_fit.fittedvalues

# 5. Inverse Transform back to 0-100 Rank Space
df_timeline['hh_econ_rank'] = norm.cdf(df_timeline['imputed_continuous']) * 100

# 6. Finalize High-Frequency Pseudo Panel Format
df_timeline['data_source'] = df_timeline['avg_rank_continuous'].isna().map({True: 'Pseudo', False: 'Survey'})

# Clean up final output
df_pseudo_panel = df_timeline[['data_source', 'o_food_price_index', 'index_MoM', 'hh_econ_rank']].copy()

display(df_pseudo_panel.tail(15))

,data_source,o_food_price_index,index_MoM,hh_econ_rank
month_period,,,,
2025-05,Pseudo,2.108966,0.000491,54.360611
2025-06,Pseudo,2.118966,0.004742,53.426899
2025-07,Pseudo,2.149655,0.014483,51.280858
2025-08,Pseudo,2.216552,0.031120,47.610758
2025-09,Pseudo,2.262414,0.020691,49.911063
2025-10,Pseudo,2.278276,0.007011,52.927607
2025-11,Pseudo,2.278276,0.000000,54.468257
2025-12,Pseudo,2.283103,0.002119,54.003229
2026-01,Pseudo,2.288276,0.002266,53.971046


In [39]:
# output
df_pseudo_panel.to_csv(r'd:\GG\output\eventstudy_pseudopanel.csv')